# Proyecto 2 — Modelo C: detección clásica + btrack

**CC3084 Data Science · UVG · Semestre II 2026**
Isabella Obando · Anthony Lou · Fabian Prado · Fernando Mendoza

**btrack** (Ulicna et al., 2021) es uno de los métodos de la revisión bibliográfica del grupo (tabla 2.5):
recibe detecciones externas y las enlaza con un **filtro de Kalman** (predice dónde estará cada célula
con un modelo de velocidad constante) y una **optimización global** (programación lineal entera sobre
hipótesis de enlace, inicio, fin y división de trayectorias). No requiere entrenamiento.

**Diseño del experimento.** El modelo C usa **exactamente las mismas detecciones que el modelo A**
(diferencia de gaussianas con los parámetros validados de la v3). Lo único que cambia es el enlace:

| | Detección | Enlace |
|---|---|---|
| Modelo A | DoG + máximos locales | asignación lineal entre instantes consecutivos, con corrección de deriva |
| **Modelo C** | **la misma** | **filtro de Kalman 3D + optimización global de hipótesis (btrack)** |

Así, cualquier diferencia en el puntaje se atribuye al método de enlace. Es la misma pregunta que se
hace el modelo base de la competencia (modelo B) cuando agrega una optimización global (ILP).

**Evaluación** con la métrica de aristas del modelo A, que coincide con la oficial, en dos conjuntos:

- las **16 muestras de validación del modelo A**: comparación directa con su 0.745, recalculado aquí;
- las **19 muestras de validación del *split 0***: las mismas que usa el modelo B, para poder comparar
  con él.

**Cómo correrlo en Kaggle:** Accelerator **None (CPU)**, **Internet On** (para instalar y compilar `btrack`) y
la competencia como *Input*. Es solo validación: no genera `submission.csv`.

## Instalación de btrack

El binario precompilado de btrack 0.7.0 exige una versión de la librería de C++ (`GLIBCXX_3.4.32`) más
nueva que la del sistema de Kaggle. Por eso se comprueba si carga y, si no, se compila su núcleo en C++
desde el código fuente (con Eigen 3.3.9, como indica su `build.sh`). La librería estándar de C++ se
enlaza de forma estática, así que el resultado no depende de la versión instalada en el sistema. Toma
alrededor de un minuto.

In [ ]:
import subprocess, sys, os, shutil, tarfile, tempfile, importlib.util
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "btrack==0.7.0"], check=True)


def btrack_carga():
    """Intenta cargar la librería C++ de btrack en un proceso aparte."""
    r = subprocess.run([sys.executable, "-c", "from btrack import libwrapper; libwrapper.get_library()"],
                       capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr)[-600:]


def correr(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"Falló: {' '.join(map(str, cmd))}\n{(r.stdout + r.stderr)[-1500:]}")


ok, detalle = btrack_carga()
if not ok or os.environ.get("BIOHUB_FORZAR_COMPILAR"):
    print("El binario precompilado de btrack no carga en este sistema; se compila desde el código fuente...")
    tmp = Path(tempfile.mkdtemp())
    correr([sys.executable, "-m", "pip", "download", "-q", "--no-deps", "--no-binary", ":all:",
            "btrack==0.7.0", "-d", str(tmp)])
    with tarfile.open(next(tmp.glob("btrack-*.tar.gz"))) as tar:
        tar.extractall(tmp, filter="data")
    fuente = next(p for p in tmp.glob("btrack-*") if p.is_dir())
    correr(["git", "clone", "-q", "--depth", "1", "--branch", "3.3.9",
            "https://gitlab.com/libeigen/eigen.git", str(fuente / "btrack" / "include" / "eigen")])
    for d in ("libs", "obj"):
        (fuente / "btrack" / d).mkdir(exist_ok=True)
    correr(["make", f"-j{os.cpu_count() or 1}", "XLDFLAGS=-Wl,--no-undefined -static-libstdc++ -static-libgcc"],
           cwd=fuente)
    destino = Path(importlib.util.find_spec("btrack").origin).parent / "libs" / "libtracker.so"
    shutil.copy(fuente / "btrack" / "libs" / "libtracker.so", destino)
    ok, detalle = btrack_carga()
    if not ok:
        raise RuntimeError(f"btrack sigue sin cargar después de compilarlo:\n{detalle}")
    print("Compilado e instalado:", destino)

import btrack
print("btrack", btrack.__version__, "listo")

## Modelo A: configuración, lectura, detección, enlace y métrica

Las mismas celdas de `modelos/modelo_A/modelo_A_v3.ipynb`, copiadas sin cambios.

In [ ]:
import os, io, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from joblib import Parallel, delayed

COMPETITION = "biohub-cell-tracking-during-development"
ES_RERUN = bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
VALIDAR = not ES_RERUN
N_JOBS = os.cpu_count() or 1


def encontrar_datos(raiz=Path("/kaggle/input")):
    """Carpeta de la competencia: la que contiene test/ (y train/) con volúmenes .zarr."""
    if os.environ.get("BIOHUB_DATA"):
        return Path(os.environ["BIOHUB_DATA"])
    candidatos = [raiz / "competitions" / COMPETITION, raiz / COMPETITION]
    for prof in range(1, 5):                       # test/ a 1 – 4 niveles de profundidad
        candidatos += [p.parent for p in sorted(raiz.glob("/".join(["*"] * prof)))
                       if p.name == "test" and p.is_dir()]
    for c in candidatos:
        if (c / "test").is_dir() and any((c / "test").glob("*.zarr")):
            return c
    contenido = sorted(str(p.relative_to(raiz)) for p in raiz.glob("*/*")) if raiz.exists() else []
    raise FileNotFoundError(
        "No se encontró la competencia en /kaggle/input. En el panel derecho, sección 'Input', "
        "agregar la competencia: + Add Input -> Competitions -> 'Biohub - Cell Tracking During Development'.\n"
        f"Contenido actual de {raiz} (2 niveles): {contenido[:30] or 'vacío'}")


DATA_DIR = encontrar_datos()
TRAIN_DIR, TEST_DIR = DATA_DIR / "train", DATA_DIR / "test"
SALIDA = Path(os.environ.get("BIOHUB_SALIDA", "/kaggle/working"))
VOXEL_UM = np.array([1.625, 0.40625, 0.40625])     # (Z, Y, X) micras por vóxel

print(f"Datos: {DATA_DIR} | rerun: {ES_RERUN} | CPUs: {N_JOBS}")

In [ ]:
import io, json
from pathlib import Path
import numpy as np


def _blosc(b):
    try:
        import numcodecs
        return numcodecs.Blosc().decode(b)
    except ImportError:
        pass
    try:
        import blosc2
        return blosc2.decompress(b)
    except ImportError:
        import blosc
        return blosc.decompress(b)


def _zstd(b):
    try:
        import numcodecs
        return numcodecs.Zstd().decode(b)
    except ImportError:
        import zstandard
        return zstandard.ZstdDecompressor().stream_reader(io.BytesIO(b)).read()


def _gzip(b):
    import gzip
    return gzip.decompress(b)


DECODIFICADORES = {"blosc": _blosc, "zstd": _zstd, "gzip": _gzip, "crc32c": lambda b: bytes(b)[:-4]}


class ArregloZarr:
    """Lector mínimo de arreglos Zarr v3 (solo lectura, sin depender de la librería zarr)."""

    def __init__(self, ruta):
        self.ruta = Path(ruta)
        m = json.loads((self.ruta / "zarr.json").read_text())
        assert m.get("node_type") == "array", f"{ruta} no es un arreglo"
        self.shape = tuple(m["shape"])
        self.chunks = tuple(m["chunk_grid"]["configuration"]["chunk_shape"])
        self.fill = m.get("fill_value") or 0
        enc = m.get("chunk_key_encoding", {"name": "default"})
        self.sep = enc.get("configuration", {}).get("separator", "/" if enc["name"] == "default" else ".")
        self.prefijo = ["c"] if enc["name"] == "default" else []
        endian = "<"
        self.codecs = []
        for c in m["codecs"]:
            n = c["name"]
            if n == "bytes":
                endian = "<" if c.get("configuration", {}).get("endian", "little") == "little" else ">"
            elif n in DECODIFICADORES:
                self.codecs.append(n)
            else:
                raise NotImplementedError(f"códec no soportado: {n}")
        self.dtype = np.dtype(m["data_type"]).newbyteorder(endian)

    def _chunk(self, idx):
        f = self.ruta / self.sep.join(self.prefijo + [str(i) for i in idx])
        if not f.exists():
            return np.full(self.chunks, self.fill, dtype=self.dtype)
        b = f.read_bytes()
        for n in reversed(self.codecs):
            b = DECODIFICADORES[n](b)
        return np.frombuffer(b, dtype=self.dtype).reshape(self.chunks)

    def leer(self):
        """Arreglo completo."""
        grid = [range(-(-s // c)) for s, c in zip(self.shape, self.chunks)]
        out = np.empty(self.shape, dtype=self.dtype.newbyteorder("="))
        for idx in np.ndindex(*[len(g) for g in grid]):
            sl = tuple(slice(i * c, min((i + 1) * c, s)) for i, c, s in zip(idx, self.chunks, self.shape))
            out[sl] = self._chunk(idx)[tuple(slice(0, x.stop - x.start) for x in sl)]
        return out

    def instante(self, t):
        """Volumen (Z, Y, X) del instante t, asumiendo chunks de un instante en el eje 0."""
        assert self.chunks[0] == 1 and self.chunks[1:] == self.shape[1:], "se esperan chunks (1, Z, Y, X)"
        return self._chunk((t, 0, 0, 0))[0].astype(self.dtype.newbyteorder("="))



def leer_grafo(geff):
    """Grafo de referencia .geff -> (DataFrame de nodos, arreglo de aristas, nodos estimados)."""
    geff = Path(geff)
    ids = ArregloZarr(geff / "nodes" / "ids").leer()
    nodos = pd.DataFrame({"node_id": ids.astype(np.int64)})
    for c in "tzyx":
        nodos[c] = ArregloZarr(geff / "nodes" / "props" / c / "values").leer().astype(float)
    aristas = ArregloZarr(geff / "edges" / "ids").leer().astype(np.int64).reshape(-1, 2)
    attrs = json.loads((geff / "zarr.json").read_text()).get("attributes", {})

    def buscar(d, k):
        if isinstance(d, dict):
            if k in d:
                return d[k]
            for v in d.values():
                r = buscar(v, k)
                if r is not None:
                    return r
        return None

    return nodos, aristas, buscar(attrs, "estimated_number_of_nodes")

In [ ]:
def normalizar(v):
    lo, hi = np.percentile(v[::2, ::4, ::4], [1, 99.9])
    return np.clip((v.astype(np.float32) - lo) / max(hi - lo, 1e-6), 0, 1)


def reducir_xy(v, f):
    if f == 1:
        return v
    Z, Y, X = v.shape
    return v[:, :Y // f * f, :X // f * f].reshape(Z, Y // f, f, X // f, f).mean(axis=(2, 4))


def filtro_maximo(dog, separacion, vox):
    """Máximo en el vecindario 'elipsoide D' o 'caja D' (D en micras)."""
    forma, d = separacion.split()
    r = np.maximum(1, np.round(float(d) / vox)).astype(int)
    if forma == "caja":
        return ndimage.maximum_filter(dog, size=2 * r + 1, mode="nearest")
    zz, yy, xx = np.ogrid[-r[0]:r[0] + 1, -r[1]:r[1] + 1, -r[2]:r[2] + 1]
    huella = (zz / r[0]) ** 2 + (yy / r[1]) ** 2 + (xx / r[2]) ** 2 <= 1
    return ndimage.maximum_filter(dog, footprint=huella, mode="nearest")


def columna(separacion):
    return "max_" + separacion.replace(" ", "_")


def detectar_volumen(v, p, umbral, separaciones):
    """Máximos locales estrictos para el primer vecindario de la lista (el más permisivo), con una
    columna por vecindario que indica si el candidato también es máximo en él."""
    f = p["reduccion_xy"]
    vr = reducir_xy(normalizar(v), f)
    vox = VOXEL_UM * np.array([1, f, f])
    s1 = p["sigma_um"] / vox
    dog = ndimage.gaussian_filter(vr, s1) - ndimage.gaussian_filter(vr, s1 * p["k_dog"])
    pico = (dog == filtro_maximo(dog, separaciones[0], vox)) & (dog > umbral)
    idx = np.argwhere(pico)
    marcas = {columna(s): dog[pico] == filtro_maximo(dog, s, vox)[pico] for s in separaciones}
    pos = idx.astype(float)
    pos[:, 1:] = pos[:, 1:] * f + (f - 1) / 2          # volver a la rejilla original
    return pos, dog[pico], marcas


def detectar_muestra(zarr_dir, p, umbral, separaciones=None):
    """Detecciones de todos los instantes -> (DataFrame t, z, y, x, resp, max_*; n_t)."""
    separaciones = separaciones or [p["separacion"]]
    arr = ArregloZarr(Path(zarr_dir) / "0")
    filas = []
    for t in range(arr.shape[0]):
        pos, resp, marcas = detectar_volumen(arr.instante(t), p, umbral, separaciones)
        filas.append(pd.DataFrame({"t": t, "z": pos[:, 0], "y": pos[:, 1], "x": pos[:, 2], "resp": resp, **marcas}))
    return pd.concat(filas, ignore_index=True), arr.shape[0]

In [ ]:
def asignar(D, radio):
    if D.size == 0:
        return np.empty((0, 2), int)
    C = np.where(D <= radio, D, 1e6)
    f, c = linear_sum_assignment(C)
    ok = C[f, c] < 1e6
    return np.column_stack([f[ok], c[ok]])


def enlazar(det, n_t, p):
    """det: detecciones (t, z, y, x, resp, max_*) -> (nodos con node_id, aristas source/target)."""
    ok = det["resp"] >= p["umbral"]
    if columna(p["separacion"]) in det:
        ok &= det[columna(p["separacion"])]
    det = det[ok].reset_index(drop=True)
    det["node_id"] = np.arange(1, len(det) + 1)
    pos = det[["z", "y", "x"]].to_numpy() * VOXEL_UM
    por_t = {t: g.index.to_numpy() for t, g in det.groupby("t")}
    vacio = np.empty(0, int)
    src, dst = [], []
    for t in range(n_t - 1):
        a, b = por_t.get(t, vacio), por_t.get(t + 1, vacio)
        if len(a) == 0 or len(b) == 0:
            continue
        P0, P1 = pos[a], pos[b]
        D = cdist(P0, P1)
        pares = asignar(D, p["radio_um"])
        if p["deriva"] == "dos_pasos" and len(pares) >= 5:
            deriva = np.median(P1[pares[:, 1]] - P0[pares[:, 0]], axis=0)
            D = cdist(P0 + deriva, P1)
            pares = asignar(D, p["radio_um"])
        src += list(a[pares[:, 0]]); dst += list(b[pares[:, 1]])

        if p["divisiones"] and len(pares):
            hijo = dict(zip(pares[:, 0], pares[:, 1]))
            libres = np.setdiff1d(np.arange(len(b)), pares[:, 1])
            usadas = set()
            for j in libres:
                cand = [(D[i, j], i) for i in hijo if i not in usadas and D[i, j] <= p["radio_div_um"]]
                if cand:
                    _, i = min(cand)
                    usadas.add(i)
                    src.append(a[i]); dst.append(b[j])

    aristas = pd.DataFrame({"source_id": det["node_id"].to_numpy()[src],
                            "target_id": det["node_id"].to_numpy()[dst]})
    return det, aristas

In [ ]:
def emparejar(pred, gt, max_um=7.0):
    """Diccionario node_id predicho -> node_id de referencia, por instante."""
    m = {}
    for t, g in gt.groupby("t"):
        pr = pred[pred["t"] == t]
        if len(pr) == 0:
            continue
        D = cdist(pr[["z", "y", "x"]].to_numpy() * VOXEL_UM, g[["z", "y", "x"]].to_numpy() * VOXEL_UM)
        for i, j in asignar(D, max_um):
            m[pr["node_id"].iat[i]] = g["node_id"].iat[j]
    return m


def metrica_aristas(pred_nodos, pred_aristas, gt_nodos, gt_aristas, n_est):
    m = emparejar(pred_nodos, gt_nodos)
    gt_set = set(map(tuple, gt_aristas.tolist()))
    sale, entra = {}, {}
    for s, d in gt_set:
        sale.setdefault(s, set()).add(d)
        entra.setdefault(d, set()).add(s)
    tp_set, fp = set(), 0
    for s, d in zip(pred_aristas["source_id"], pred_aristas["target_id"]):
        ms, md_ = m.get(s), m.get(d)
        if ms is not None and md_ is not None and (ms, md_) in gt_set:
            tp_set.add((ms, md_))
        elif (md_ is not None and entra.get(md_, set()) - {ms}) or \
             (ms is not None and sale.get(ms, set()) - {md_}):
            fp += 1
    tp, fn = len(tp_set), len(gt_set) - len(tp_set)
    j = tp / max(tp + fp + fn, 1)
    n_pred = len(pred_nodos)
    adj = max(0.0, j * (1 - 0.1 * (n_pred - n_est) / n_est)) if n_est else j
    recall_nodos = len(set(m.values())) / max(len(gt_nodos), 1)
    return dict(tp=tp, fp=fp, fn=fn, jaccard=j, jaccard_ajustado=adj, n_pred=n_pred, n_est=n_est,
                recall_nodos=recall_nodos)


def agregar(res):
    r = pd.DataFrame(res)
    w = r["tp"] + r["fp"] + r["fn"]
    return pd.Series({"jaccard_ajustado": (r["jaccard_ajustado"] * w).sum() / w.sum(),
                      "jaccard": r["tp"].sum() / w.sum(),
                      "recall_nodos": r["recall_nodos"].mean(),
                      "pred/estimados": r["n_pred"].sum() / r["n_est"].sum(),
                      "tp": r["tp"].sum(), "fp": r["fp"].sum(), "fn": r["fn"].sum()})

In [ ]:
# Parámetros validados del modelo A (v1 = v3): σ 2.5 µm, elipsoide de 3 µm, umbral 0.01, radio 8.4 µm
PARAMS_A = {'reduccion_xy': 2, 'sigma_um': 2.5, 'k_dog': 1.6, 'separacion': 'elipsoide 3.0', 'umbral': 0.01,
            'radio_um': 8.4, 'deriva': 'dos_pasos', 'divisiones': False, 'radio_div_um': 8.0}

## Modelo C: enlace con btrack

- Las posiciones se pasan **en micras**, así el modelo de movimiento es isotrópico (la anisotropía 4:1
  del vóxel ya está corregida).
- **Modelo de movimiento:** velocidad constante en 3D (estado = posición + velocidad). `sigma_mov`
  controla cuánto puede cambiar la velocidad entre instantes; es el parámetro que se ajusta.
- **Radio de búsqueda:** 8.4 µm, el p99 del desplazamiento del EDA (el mismo del modelo A).
- **Hipótesis de la optimización global:** falso positivo, inicio, fin y enlace de fragmentos de
  trayectoria. La de **división** se prueba con y sin ella, igual que en el modelo A.
- La competencia solo tiene aristas entre instantes consecutivos. Si btrack salta un instante (rellena
  un hueco), ese enlace se descarta.

In [ ]:
import logging
from btrack.utils import objects_from_dict

PARAMS_C = {**PARAMS_A, "sigma_mov": 2.0, "divisiones": False, "max_lost": 1}


def config_btrack(p):
    I, Z0 = np.eye(3), np.zeros((3, 3))
    hip = ["P_FP", "P_init", "P_term", "P_link"] + (["P_branch"] if p["divisiones"] else [])
    return {
        "name": "biohub", "max_search_radius": p["radio_um"], "verbose": False,
        "motion_model": {
            "name": "velocidad_constante_3d", "measurements": 3, "states": 6, "dt": 1.0,
            "accuracy": 2.0, "max_lost": p["max_lost"], "prob_not_assign": 0.1,
            "A": np.block([[I, I], [Z0, I]]).ravel().tolist(),                  # x' = x + v
            "H": np.hstack([I, Z0]).ravel().tolist(),                          # se observa la posición
            "P": {"sigma": 10.0, "matrix": np.eye(6).ravel().tolist()},
            "G": {"sigma": p["sigma_mov"], "matrix": [0.5, 0.5, 0.5, 1, 1, 1]},  # ruido del movimiento
            "R": {"sigma": 1.0, "matrix": np.eye(3).ravel().tolist()},         # ruido de medición (~1 µm)
        },
        "hypothesis_model": {
            "name": "hipotesis", "hypotheses": hip, "lambda_time": 5.0, "lambda_dist": 3.0,
            "lambda_link": 5.0, "lambda_branch": 20.0, "eta": 1e-10, "theta_dist": 5.0, "theta_time": 3.0,
            "dist_thresh": p["radio_um"], "time_thresh": 2, "apop_thresh": 5,
            "segmentation_miss_rate": 0.1, "apoptosis_rate": 0.001, "relax": True,
        },
    }


def enlazar_btrack(det, n_t, forma, p):
    """Mismo filtrado de detecciones que el modelo A; enlace con btrack. -> (nodos, aristas)."""
    logging.getLogger("btrack").setLevel(logging.WARNING)
    ok = det["resp"] >= p["umbral"]
    if columna(p["separacion"]) in det:
        ok &= det[columna(p["separacion"])]
    det = det[ok].reset_index(drop=True)
    det["node_id"] = np.arange(1, len(det) + 1)
    vacio = pd.DataFrame({"source_id": pd.Series(dtype=np.int64), "target_id": pd.Series(dtype=np.int64)})
    if len(det) == 0:
        return det, vacio
    um = det[["z", "y", "x"]].to_numpy() * VOXEL_UM
    objetos = objects_from_dict({"ID": np.arange(len(det)), "t": det["t"].to_numpy().astype(int),
                                 "x": um[:, 2], "y": um[:, 1], "z": um[:, 0]})
    lim = np.array(forma) * VOXEL_UM
    with btrack.BayesianTracker(verbose=False) as tr:
        tr.configure(config_btrack(p))
        tr.volume = ((0, lim[2]), (0, lim[1]), (0, lim[0]))
        tr.append(objetos)
        tr.track(step_size=100)
        tr.optimize(msg_lev="GLP_MSG_OFF", tm_lim=60_000)   # conserva el límite de 60 s por defecto
        tracks = tr.tracks

    t_obj = det["t"].to_numpy()
    src, dst, primero, ultimo = [], [], {}, {}
    for trk in tracks:
        refs = [r for r in trk.refs if r >= 0]           # refs < 0: objetos de relleno de btrack
        for a, b in zip(refs[:-1], refs[1:]):
            if t_obj[b] - t_obj[a] == 1:
                src.append(a); dst.append(b)
        if refs:
            primero[trk.ID], ultimo[trk.ID] = refs[0], refs[-1]
    if p["divisiones"]:
        for trk in tracks:
            for hija in trk.children:
                a, b = ultimo.get(trk.ID), primero.get(hija)
                if a is not None and b is not None and t_obj[b] - t_obj[a] == 1:
                    src.append(a); dst.append(b)
    ids = det["node_id"].to_numpy()
    return det, (pd.DataFrame({"source_id": ids[src], "target_id": ids[dst]}) if src else vacio)

## Muestras de validación y detecciones

Las detecciones se calculan **una sola vez** por muestra (es la parte costosa) y las usan los dos
enlaces.

In [ ]:
stems = sorted(p.name[:-5] for p in TRAIN_DIR.glob("*.geff"))

# Validación del modelo A: 8 muestras por embrión, mismo sorteo que modelo_A_v3.ipynb
rng = np.random.default_rng(0)
por_emb = {}
for s in stems:
    por_emb.setdefault(s.split("_")[0], []).append(s)
VAL_A = sorted(s for e, ss in por_emb.items() for s in rng.choice(ss, min(8, len(ss)), replace=False))

# Validación del split 0 (la del modelo B): random.Random(0) sobre las muestras con .zarr y .geff, 10%
import random
pares = sorted(s for s in stems if (TRAIN_DIR / f"{s}.zarr").exists())
random.Random(0).shuffle(pares)
VAL_B = sorted(pares[:max(1, len(pares) // 10)])

TODAS = sorted(set(VAL_A) | set(VAL_B))
print(f"VAL_A: {len(VAL_A)} | VAL_B (split 0): {len(VAL_B)} | en común: {len(set(VAL_A) & set(VAL_B))} | total: {len(TODAS)}")


def detectar_con_forma(s):
    z = TRAIN_DIR / f"{s}.zarr"
    det, n_t = detectar_muestra(z, PARAMS_A, PARAMS_A["umbral"])
    return det, n_t, ArregloZarr(z / "0").shape[1:]


t0 = time.time()
DET = dict(zip(TODAS, Parallel(n_jobs=N_JOBS)(delayed(detectar_con_forma)(s) for s in TODAS)))
GT = {s: leer_grafo(TRAIN_DIR / f"{s}.geff") for s in TODAS}
print(f"Detecciones: {time.time() - t0:.0f} s para {len(TODAS)} muestras")

## Ajuste de btrack

Se ajustan `sigma_mov` (cuánto puede variar la velocidad: 1 o 3 µm) y las divisiones en las **16 muestras del
modelo A**, las mismas con que se ajustó el modelo A. Las 19 del *split 0* quedan como verificación.

In [ ]:
def evaluar(metodo, p, muestras):
    def una(s, d, gt):
        det, n_t, forma = d
        gn, ga, n_est = gt
        t0 = time.time()
        nodos, aristas = enlazar_btrack(det, n_t, forma, p) if metodo == "C" else enlazar(det, n_t, p)
        r = metrica_aristas(nodos, aristas, gn, ga, n_est)
        r.update(sample=s, seg_enlace=time.time() - t0)
        return r
    r = pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(una)(s, DET[s], GT[s]) for s in muestras))
    r["embryo_id"] = r["sample"].str.split("_").str[0]
    return r


def resumen(r):
    filas = {"total": agregar(r)}
    filas.update({e: agregar(g) for e, g in r.groupby("embryo_id")})
    t = pd.DataFrame(filas).T[["jaccard_ajustado", "recall_nodos", "pred/estimados", "tp", "fp", "fn"]]
    t["seg_enlace_por_muestra"] = [r["seg_enlace"].mean()] + [g["seg_enlace"].mean() for _, g in r.groupby("embryo_id")]
    return t


# btrack crece más que linealmente con el número de detecciones (~50 s con 25 mil), así que la
# grilla es pequeña: 2 valores de ruido de movimiento x divisiones sí/no.
filas, CACHE_C = [], {}
for sigma_mov in [1.0, 3.0]:
    for div in [False, True]:
        t0 = time.time()
        p = {**PARAMS_C, "sigma_mov": sigma_mov, "divisiones": div}
        CACHE_C[(sigma_mov, div)] = r = evaluar("C", p, VAL_A)
        a = agregar(r)
        filas.append({"sigma_mov": sigma_mov, "divisiones": div, **a.to_dict()})
        print(f"sigma_mov={sigma_mov} divisiones={div}: J ajustado {a['jaccard_ajustado']:.4f} "
              f"({time.time() - t0:.0f} s)", flush=True)
AJUSTE_C = pd.DataFrame(filas)
display(AJUSTE_C.round(4))
mejor = AJUSTE_C.loc[AJUSTE_C["jaccard_ajustado"].idxmax()]
PARAMS_C = {**PARAMS_C, "sigma_mov": float(mejor["sigma_mov"]), "divisiones": bool(mejor["divisiones"])}
print(f"-> mejor: sigma_mov = {PARAMS_C['sigma_mov']}, divisiones = {PARAMS_C['divisiones']}")

## Comparación A vs. C

Mismas detecciones, mismas muestras, misma métrica: lo único distinto es el enlace.

In [ ]:
RES, RAW = {}, {}
for nombre, muestras in [("VAL_A (16)", VAL_A), ("VAL_B split 0 (19)", VAL_B)]:
    for m, p in [("A", PARAMS_A), ("C", PARAMS_C)]:
        if m == "C" and muestras is VAL_A:          # ya calculado durante el ajuste
            r = CACHE_C[(PARAMS_C["sigma_mov"], PARAMS_C["divisiones"])]
        else:
            r = evaluar(m, p, muestras)
        RAW[(nombre, m)] = r
        RES[(nombre, m)] = resumen(r)

for nombre in ["VAL_A (16)", "VAL_B split 0 (19)"]:
    print(f"\n=== {nombre} ===")
    for m, etiqueta in [("A", "A · asignación lineal"), ("C", "C · btrack")]:
        print(f"\nModelo {etiqueta}:")
        display(RES[(nombre, m)].round(4))

tabla = pd.DataFrame({
    (nombre, grupo): {m: RES[(nombre, m)].loc[grupo, "jaccard_ajustado"] for m in ["A", "C"]}
    for nombre in ["VAL_A (16)", "VAL_B split 0 (19)"] for grupo in RES[(nombre, "A")].index}).T
tabla["C − A"] = tabla["C"] - tabla["A"]
print("\nJaccard de aristas ajustado:")
display(tabla.round(4))

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(tabla))
ax.bar(x - .2, tabla["A"], .4, label="A · asignación lineal", color="tab:blue")
ax.bar(x + .2, tabla["C"], .4, label="C · btrack", color="tab:green")
ax.set_xticks(x, [f"{n.split(' ')[0]}\n{g}" for n, g in tabla.index])
ax.set_ylabel("Jaccard de aristas ajustado"); ax.legend(); ax.set_ylim(0, 1)
ax.set_title("Mismas detecciones, distinto enlace")
plt.tight_layout(); plt.show()

### Lectura de los resultados

> ⏳ **Interpretación pendiente**: completar con los resultados de la corrida en Kaggle.

## Exportar resultados para la aplicación

Se guardan en `resultados_modelos.zip` (descargarlo desde *Output* y descomprimirlo en `app/`):

- `resultados_por_muestra.csv`: métricas de los modelos A y C en cada muestra de los dos conjuntos de
  validación (TP, FP, FN, Jaccard, recall, nodos predichos y tiempo de enlace).
- `ajuste_modelo_C.csv`: la búsqueda de parámetros de btrack.
- `parametros.json`: los parámetros finales de ambos modelos.

In [ ]:
import shutil
SAL = SALIDA / "resultados_modelos"
SAL.mkdir(parents=True, exist_ok=True)
por_muestra = pd.concat([r.assign(conjunto=n, modelo=m) for (n, m), r in RAW.items()], ignore_index=True)
por_muestra.to_csv(SAL / "resultados_por_muestra.csv", index=False)
AJUSTE_C.to_csv(SAL / "ajuste_modelo_C.csv", index=False)
(SAL / "parametros.json").write_text(json.dumps({"PARAMS_A": PARAMS_A, "PARAMS_C": PARAMS_C}, indent=2, default=str))
zip_ = shutil.make_archive(str(SALIDA / "resultados_modelos"), "zip", SAL.parent, SAL.name)
print(f"{zip_} | {len(por_muestra)} filas por muestra")
por_muestra.groupby(["conjunto", "modelo"])["jaccard_ajustado"].mean().round(4)